# Single Image Dehazing with a Convolutional Autoencoder

Train a supervised encoder–decoder to reconstruct clear images from paired hazy images.
The network processes each color channel independently using shared weights.

The uploaded notebook recorded **25.4685 dB mean PSNR on 55 training image pairs**.
It did not use a held-out test set or record an SSIM score. This is a historical
reconstruction result, not a benchmark test result or a measurement of this cleaned version.

## 1. Setup

Install the packages listed in `requirements.txt`, then run the cells in order.
Training uses CUDA when available and otherwise runs on CPU.

In [ ]:
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATA_DIR = Path("data/Dense_Haze_NTIRE19")
GT_DIR = DATA_DIR / "GT"
HAZY_DIR = DATA_DIR / "hazy"
OUTPUT_DIR = Path("outputs")
CHECKPOINT_PATH = OUTPUT_DIR / "dehazing_autoencoder.pt"
CUSTOM_IMAGE_PATH = None  # Set to Path("data/my_hazy_image.jpg") for inference.

IMAGE_SIZE = 256
EPOCHS = 1000
BATCH_SIZE = 1
LEARNING_RATE = 5e-4
SEED = 42
TRAIN_FRACTION = 1.0  # Original protocol. Set to 0.8 for an image-level holdout.
RUN_TRAINING = True  # Set to False to use an existing compatible checkpoint.

if IMAGE_SIZE <= 0 or IMAGE_SIZE % 4:
    raise ValueError("IMAGE_SIZE must be positive and divisible by 4.")
if EPOCHS <= 0 or BATCH_SIZE <= 0 or LEARNING_RATE <= 0:
    raise ValueError("Training parameters must be positive.")
if not 0 < TRAIN_FRACTION <= 1:
    raise ValueError("TRAIN_FRACTION must be in (0, 1].")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {device}")


## 2. Load paired images

Match files by image ID (`01_hazy.png` and `01_GT.png`), convert to RGB,
resize to 256 × 256, and scale pixel values to [0, 1].
The optional holdout splits image pairs before separating color channels.
A random holdout is not the official NTIRE benchmark split.

In [ ]:
def find_pairs(hazy_dir, gt_dir):
    hazy_dir, gt_dir = Path(hazy_dir), Path(gt_dir)
    if not hazy_dir.is_dir() or not gt_dir.is_dir():
        raise FileNotFoundError("Set HAZY_DIR and GT_DIR to the dataset directories.")
    hazy_files = {p.name.removesuffix("_hazy.png"): p for p in hazy_dir.glob("*_hazy.png")}
    gt_files = {p.name.removesuffix("_GT.png"): p for p in gt_dir.glob("*_GT.png")}
    if not hazy_files or not gt_files:
        raise ValueError("Expected files named <id>_hazy.png and <id>_GT.png.")
    if hazy_files.keys() != gt_files.keys():
        missing_gt = sorted(hazy_files.keys() - gt_files.keys())
        missing_hazy = sorted(gt_files.keys() - hazy_files.keys())
        raise ValueError(f"Unmatched pairs. Missing GT: {missing_gt}; missing hazy: {missing_hazy}")
    return [(key, hazy_files[key], gt_files[key]) for key in sorted(hazy_files)]


def load_rgb(path, image_size):
    with Image.open(path) as image:
        image = image.convert("RGB").resize(
            (image_size, image_size), Image.Resampling.BILINEAR
        )
        return np.asarray(image, dtype=np.float32) / 255.0


pairs = find_pairs(HAZY_DIR, GT_DIR)
image_ids = [item[0] for item in pairs]
hazy_images = np.stack([load_rgb(item[1], IMAGE_SIZE) for item in tqdm(pairs, desc="Hazy images")])
clear_images = np.stack([load_rgb(item[2], IMAGE_SIZE) for item in tqdm(pairs, desc="Clear images")])

if TRAIN_FRACTION == 1.0:
    train_indices = np.arange(len(pairs))
    eval_indices = train_indices.copy()
    evaluation_label = "Training-set reconstruction"
else:
    if len(pairs) < 2:
        raise ValueError("At least two image pairs are needed for a holdout.")
    indices = np.random.default_rng(SEED).permutation(len(pairs))
    n_train = max(1, min(len(pairs) - 1, int(len(pairs) * TRAIN_FRACTION)))
    train_indices, eval_indices = indices[:n_train], indices[n_train:]
    evaluation_label = "Held-out reconstruction (random image-level split)"

print(f"Paired images: {len(pairs)}; training: {len(train_indices)}; evaluation: {len(eval_indices)}")
print(evaluation_label)


In [ ]:
def plot_triplet(hazy, clear, prediction=None):
    images = [(hazy, "Hazy"), (clear, "Ground truth")]
    if prediction is not None:
        images.append((prediction, "Dehazed"))
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 4))
    for ax, (image, title) in zip(axes, images):
        ax.imshow(np.clip(image, 0, 1))
        ax.set_title(title)
        ax.axis("off")
    fig.tight_layout()
    plt.show()


plot_triplet(hazy_images[0], clear_images[0])


## 3. Model

Each input is one color channel of size 1 × 256 × 256. The encoder uses
seven 3 × 3 convolutions, ReLU activations, batch normalization, and two
max-pooling operations to produce a 256 × 64 × 64 feature map.
The decoder uses seven transposed convolutions, with two stride-2 layers
restoring the original resolution. Its output activation is ReLU.

RGB images are reconstructed by applying the same network to their three
channels. There are no skip connections, attention layers, or dense layers.

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
        )

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        return out

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 3, 2, 1, 1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.ConvTranspose2d(128, 128, 3, 1, 1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.ConvTranspose2d(128, 64, 3, 1, 1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.ConvTranspose2d(64, 64, 3, 1, 1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
        )
        self.layer2 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, 1, 1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.ConvTranspose2d(32, 32, 3, 1, 1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.ConvTranspose2d(32, 1, 3, 2, 1, 1),
            nn.ReLU(),
        )

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        return out


class DehazingAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        return self.decoder(self.encoder(x))


model = DehazingAutoencoder().to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Trainable parameters: {parameter_count:,}")


## 4. Training

Use Adam with a learning rate of 0.0005 and mean squared error loss.
Channel samples remain paired when shuffled. Log the average loss over
each epoch and save model weights, configuration, and image IDs.

In [ ]:
def channel_tensor(images):
    nchw = np.ascontiguousarray(images.transpose(0, 3, 1, 2))
    return torch.from_numpy(nchw).reshape(-1, 1, images.shape[1], images.shape[2])


train_dataset = TensorDataset(
    channel_tensor(hazy_images[train_indices]),
    channel_tensor(clear_images[train_indices]),
)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
losses = []

if RUN_TRAINING:
    for epoch in tqdm(range(EPOCHS), desc="Training"):
        model.train()
        total_loss = 0.0
        for hazy_batch, clear_batch in train_loader:
            hazy_batch = hazy_batch.to(device)
            clear_batch = clear_batch.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(hazy_batch), clear_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * hazy_batch.size(0)
        losses.append(total_loss / len(train_dataset))

    torch.save({
        "model_state_dict": model.state_dict(),
        "config": {
            "image_size": IMAGE_SIZE, "epochs": EPOCHS,
            "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
            "seed": SEED, "train_fraction": TRAIN_FRACTION,
        },
        "train_ids": [image_ids[i] for i in train_indices],
        "eval_ids": [image_ids[i] for i in eval_indices],
        "losses": losses,
    }, CHECKPOINT_PATH)
    print(f"Saved checkpoint: {CHECKPOINT_PATH}")
elif not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError("No checkpoint found. Set RUN_TRAINING=True first.")


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
if checkpoint["config"]["image_size"] != IMAGE_SIZE:
    raise ValueError("IMAGE_SIZE differs from the saved training configuration.")
if (checkpoint["train_ids"] != [image_ids[i] for i in train_indices]
        or checkpoint["eval_ids"] != [image_ids[i] for i in eval_indices]):
    raise ValueError("Current dataset split differs from the saved checkpoint. Restore its configuration.")
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
losses = checkpoint["losses"]

if losses:
    plt.figure(figsize=(7, 4))
    plt.plot(range(1, len(losses) + 1), losses)
    plt.xlabel("Epoch")
    plt.ylabel("Mean MSE")
    plt.title("Training loss")
    plt.tight_layout()
    plt.show()


## 5. Reconstruction and evaluation

PSNR is calculated for each complete RGB image using a peak intensity of 1,
then averaged across images. Predictions are not clipped for PSNR, matching
the original metric convention; only plots and exported images are clipped.
The model runs in evaluation mode with gradients disabled.

In [ ]:
def predict_rgb(model, image, device):
    channels = channel_tensor(image[np.newaxis])
    predictions = []
    with torch.inference_mode():
        for channel in channels.split(BATCH_SIZE):
            predictions.append(model(channel.to(device)).cpu())
    output = torch.cat(predictions, dim=0).squeeze(1)
    return output.permute(1, 2, 0).numpy()


def image_psnr(reference, prediction):
    error = reference.astype(np.float64) - prediction.astype(np.float64)
    mse = np.mean(error ** 2)
    return float("inf") if mse == 0 else float(10 * np.log10(1.0 / mse))


records = []
reconstructed_images = []
for idx in tqdm(eval_indices, desc="Evaluating"):
    prediction = predict_rgb(model, hazy_images[idx], device)
    reconstructed_images.append(prediction)
    records.append({
        "image_id": image_ids[idx],
        "hazy_psnr_db": image_psnr(clear_images[idx], hazy_images[idx]),
        "dehazed_psnr_db": image_psnr(clear_images[idx], prediction),
    })

mean_hazy_psnr = np.mean([row["hazy_psnr_db"] for row in records])
mean_dehazed_psnr = np.mean([row["dehazed_psnr_db"] for row in records])
print(evaluation_label)
print(f"Hazy input mean PSNR: {mean_hazy_psnr:.4f} dB")
print(f"Dehazed mean PSNR: {mean_dehazed_psnr:.4f} dB")

import csv
with (OUTPUT_DIR / "psnr_by_image.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(records[0]))
    writer.writeheader()
    writer.writerows(records)

for position in range(min(3, len(eval_indices))):
    idx = eval_indices[position]
    plot_triplet(hazy_images[idx], clear_images[idx], reconstructed_images[position])


## 6. Custom image

Set `CUSTOM_IMAGE_PATH` in the configuration cell. Inference uses the same
RGB conversion, resizing, [0, 1] scaling, and channel processing as training.
No quality score is computed without a matching clear reference image.

In [ ]:
if CUSTOM_IMAGE_PATH is not None:
    custom_image = load_rgb(Path(CUSTOM_IMAGE_PATH), IMAGE_SIZE)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    custom_prediction = predict_rgb(model, custom_image, device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    output_array = np.rint(np.clip(custom_prediction, 0, 1) * 255).astype(np.uint8)
    output_path = OUTPUT_DIR / "dehazed_custom.png"
    Image.fromarray(output_array).save(output_path)
    print(f"Saved: {output_path}; inference time: {elapsed:.3f} seconds")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, image, title in zip(axes, [custom_image, custom_prediction], ["Hazy", "Dehazed"]):
        ax.imshow(np.clip(image, 0, 1))
        ax.set_title(title)
        ax.axis("off")
    fig.tight_layout()
    plt.show()
else:
    print("Set CUSTOM_IMAGE_PATH to dehaze your own image.")


## Result interpretation

The original notebook recorded 25.4685 dB on the training images, with no
held-out test evaluation. It contained no computed SSIM or timed inference
result. The cleaned version fixes preprocessing and inference behavior,
so new measurements must be obtained by running it with the actual data.

Dataset reference: [Dense-Haze (NTIRE 2019)](https://data.vision.ee.ethz.ch/cvl/ntire19/dense-haze/).